In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# read the bronze tabel
df = spark.table("workspace.bronze.flights_raw")

# clean the spaces in columns
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        col_ref = F.col(f"`{field.name}`")

        df = df.withColumn(
            field.name,
            F.when(
                F.trim(col_ref) == "",
                F.lit(None).cast("string")
            ).otherwise(F.trim(col_ref))
        )

# save this modifided version into silver 
(
    df.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable("workspace.silver.flights_clean")
)

print("Silver table created successfully.")


In [0]:
%sql

-- Create the Silver table from the Bronze table
CREATE OR REPLACE TABLE workspace.silver.flights_clean
USING DELTA
AS
SELECT
    * REPLACE (
        NULLIF(TRIM(`FlightDate`), '') AS `FlightDate`
    )
FROM workspace.bronze.flights_raw;


القيم الفريده 

In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT flight_number) AS unique_flight_numbers,
    SUM(CASE WHEN status = 'Unknown' THEN 1 ELSE 0 END)
        AS unknown_status_rows,
    SUM(CASE WHEN `aircraft.reg` IS NULL THEN 1 ELSE 0 END)
        AS missing_aircraft_reg,
    SUM(CASE WHEN `movement.terminal` IS NULL THEN 1 ELSE 0 END)
        AS missing_terminal
    
FROM workspace.silver.flights_clean;


اعلى destination_airport_iata له سحلا طيران

In [0]:
%sql

SELECT
    flight_type,
    `origin_airport_iata`,
    `destination_airport_iata`,
    COUNT(*) AS flight_records
FROM workspace.silver.flights_clean
GROUP BY
    flight_type,
    `origin_airport_iata`,
    `destination_airport_iata`
ORDER BY flight_records DESC
LIMIT 20;


percentage of missing value

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

df = spark.table("workspace.silver.flights_clean")


expressions = []
column_conditions = []

for i, field in enumerate(df.schema.fields):
    c = F.col(f"`{field.name}`")

   
    condition = c.isNull()

    if isinstance(field.dataType, StringType):
        condition = condition | (F.trim(c) == "")

    column_conditions.append((field.name, condition))

    expressions.append(
        F.sum(
            F.when(condition, 1).otherwise(0)
        ).alias(f"missing_{i}")
    )


counts = df.agg(*expressions).first()

total_rows = df.count()
results = []

for i, (name, condition) in enumerate(column_conditions):
    missing = counts[f"missing_{i}"] or 0

    results.append((
        name,
        int(missing),
        round(100 * missing / total_rows, 2) if total_rows else 0,
        total_rows - int(missing)
    ))

quality_df = spark.createDataFrame(
    results,
    [
        "column_name",
        "missing_count",
        "missing_percentage",
        "non_missing_count"
    ]
)

display(
    quality_df.orderBy(
        F.desc("missing_percentage")
    )
)


display the names of columns (we should put it in front after the creation of table)

In [0]:
%sql

DESCRIBE TABLE workspace.silver.flights_clean;



In [0]:
%sql

SELECT
    flight_number,
    `movement.scheduledTime.utc` AS scheduled_time_utc,
    flight_type,
    COUNT(*) AS record_count
FROM workspace.silver.flights_clean
GROUP BY
    flight_number,
    `movement.scheduledTime.utc`,
    flight_type
HAVING COUNT(*) > 1
ORDER BY record_count DESC
LIMIT 20;


check if its benifit or not

In [0]:
%sql

SELECT
    flight_number,
    `movement.scheduledTime.utc` AS scheduled_time_utc,
    flight_type,
    COUNT(*) AS record_count,
    COUNT(DISTINCT `airline.name`) AS distinct_airlines
FROM workspace.silver.flights_clean
GROUP BY
    flight_number,
    `movement.scheduledTime.utc`,
    flight_type
HAVING COUNT(*) > 1
ORDER BY record_count DESC;


check the number of airlines names

In [0]:
%sql

SELECT COUNT(DISTINCT `airline.name`) AS distinct_airlines
FROM workspace.silver.flights_clean;

In [0]:

from pyspark.sql import functions as F

df = spark.table("workspace.silver.flights_clean")

display(
    df.filter(
        (F.col("flight_number") == "SV 1023") &
        (F.col("`movement.scheduledTime.utc`").cast("string")
         .contains("2025-04-26 05:00"))
    )
)


remove the dublicated rows

In [0]:
%sql
CREATE OR REPLACE TABLE silver.flights_clean AS
WITH RankedFlights AS (
    SELECT 
        *,
        
        ROW_NUMBER() OVER (
            PARTITION BY flight_number, `movement.scheduledTime.utc`, origin_airport_icao, destination_airport_icao
            ORDER BY 
                
                CASE WHEN `movement.terminal` IS NOT NULL THEN 1 ELSE 0 END DESC,
                CASE WHEN status != 'Unknown' THEN 1 ELSE 0 END DESC,
                `movement.scheduledTime.local` DESC 
        ) as row_num
    FROM bronze.flights_raw 
)
SELECT * EXCEPT(row_num) 
FROM RankedFlights
WHERE row_num = 1;

In [0]:
%sql
SELECT * FROM silver.flights_clean LIMIT 10;

In [0]:
%sql
SELECT COUNT(*) FROM silver.flights_clean;

No need for apply any changes on the name of airlines

In [0]:
%sql
-- 
SELECT 
    `airline.name`,
    UPPER(`airline.name`) AS upper_version,
    LOWER(`airline.name`) AS lower_version,
    COUNT(*) AS count
FROM silver.flights_clean
GROUP BY `airline.name`, UPPER(`airline.name`), LOWER(`airline.name`)
ORDER BY `airline.name`;

check the names of airlines 

In [0]:
%sql
--
SELECT 
    `airline.name`,
    UPPER(`airline.name`) AS upper_version,
    LOWER(`airline.name`) AS lower_version,
    COUNT(*) AS count
FROM silver.flights_clean
GROUP BY `airline.name`, UPPER(`airline.name`), LOWER(`airline.name`)
ORDER BY `airline.name`;

check the name of   origin_airport_name

In [0]:
%sql
-- 
SELECT 
    origin_airport_name,
    COUNT(*) AS count
FROM silver.flights_clean
GROUP BY origin_airport_name
ORDER BY origin_airport_name;

The rarest airport flight

In [0]:
%sql
--
SELECT 
    destination_airport_name,
    COUNT(*) AS count
FROM silver.flights_clean
GROUP BY destination_airport_name
HAVING COUNT(*) < 5
ORDER BY count;

Check if the flight is from an airport to the same airport

In [0]:
%sql
SELECT COUNT(*) AS same_airport_flights
FROM silver.flights_clean
WHERE origin_airport_iata = destination_airport_iata;

The trip time is negative or zero

In [0]:
%sql
SELECT COUNT(*) AS invalid_duration
FROM silver.flights_clean
WHERE `movement.scheduledTime.local` <= `movement.scheduledTime.utc` 
  AND origin_airport_icao = destination_airport_icao;

Airports not included in the list IATA official

In [0]:
%sql
-- check if the code less than 3
SELECT DISTINCT origin_airport_iata
FROM silver.flights_clean
WHERE LENGTH(origin_airport_iata) != 3;

Dates in the distant future

In [0]:
%sql
SELECT COUNT(*) AS future_flights
FROM silver.flights_clean
WHERE to_timestamp(`movement.scheduledTime.utc`, 'yyyy-MM-dd HH:mmXXX') > CURRENT_TIMESTAMP();

negative value in isCargo columns

In [0]:
%sql
SELECT COUNT(*) AS negative_values
FROM silver.flights_clean
WHERE isCargo NOT IN (true, false);  -- it should be boolean

In [0]:
%sql
CREATE OR REPLACE TABLE silver.flights_clean AS
SELECT *
FROM silver.flights_clean
WHERE 
    -- 1.
    origin_airport_iata != destination_airport_iata
    -- 2. not correct code
    AND LENGTH(origin_airport_iata) = 3
    AND LENGTH(destination_airport_iata) = 3
    -- 3. datest is not in the future
    AND to_timestamp(`movement.scheduledTime.utc`, 'yyyy-MM-dd HH:mmXXX') <= DATE_ADD(CURRENT_TIMESTAMP(), 365)
    -- 4. Exclude trips with no time
    AND `movement.scheduledTime.utc` IS NOT NULL;

In [0]:
%sql
SELECT 
    destination_airport_name,
    destination_airport_iata,
    destination_airport_icao,
    COUNT(*) AS count
FROM silver.flights_clean
GROUP BY destination_airport_name, destination_airport_iata, destination_airport_icao
HAVING COUNT(*) < 5
ORDER BY count;